# Zvec: A Lightweight Embedded Vector Database for RAG Systems

Real Zvec0.8 + original MiniLM embedding model and three source documents. The final prompt is completed with a small official local Qwen2.5 instruction model. CPU only; no key, paid API or database service. This three-document example tests mechanics, not retrieval quality or large-scale ANN throughput. A flat cosine index is used for independently checkable exact ranking. Run all cells in order.

In [ ]:
import os, json, tempfile, gc, hashlib
from pathlib import Path
import numpy as np
import torch
import zvec
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
# python -m pip install zvec==0.8.0 sentence-transformers==6.2.0 transformers==5.22.0 torch==2.10.0
torch.set_num_threads(4)
torch.manual_seed(23)
cache = os.environ.get('TDB_MODEL_CACHE')
work = tempfile.TemporaryDirectory(prefix='tdb-zvec-')
root = Path(work.name)
print('Real in-process Zvec:', zvec.__version__, '; local CPU models, no paid API.')

In [ ]:
MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'
MODEL_REVISION = '1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
encoder = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device='cpu',
    cache_folder=cache, token=False, trust_remote_code=False)
documents = [
    'Zvec is an embedded vector database for AI applications.',
    'Vector databases enable semantic similarity search.',
    'Retrieval Augmented Generation improves LLM responses using external data.'
]
embeddings = encoder.encode(documents, normalize_embeddings=True, convert_to_numpy=True,
    show_progress_bar=False).astype(np.float32)
assert embeddings.shape == (3,384) and np.isfinite(embeddings).all()
np.testing.assert_allclose(np.linalg.norm(embeddings,axis=1),1.0,atol=1e-6)
print('Embedding shape:', embeddings.shape, '; revision:', MODEL_REVISION)

In [ ]:
schema = zvec.CollectionSchema(name='knowledge', fields=zvec.FieldSchema('text',zvec.DataType.STRING),
    vectors=zvec.VectorSchema('embedding',zvec.DataType.VECTOR_FP32,384,
        index_param=zvec.FlatIndexParam(metric_type=zvec.MetricType.COSINE)))
path = root/'knowledge'
db = zvec.create_and_open(str(path),schema)
db.insert([zvec.Doc(id=f'doc_{i}',vectors={'embedding':vector.tolist()},fields={'text':text})
    for i,(text,vector) in enumerate(zip(documents,embeddings))])
db.flush()
print('Stored all',len(documents),'documents with text fields and 384-dimensional vectors.')

In [ ]:
query = 'What is a vector database?'
query_vector = encoder.encode(query,normalize_embeddings=True,show_progress_bar=False).astype(np.float32)
results = db.query(zvec.Query(field_name='embedding',vector=query_vector.tolist()),
    topk=2, output_fields=['text'])
reference_distances = 1.0 - embeddings @ query_vector
expected_order = np.argsort(reference_distances)[:2]
assert [doc.id for doc in results] == [f'doc_{i}' for i in expected_order]
np.testing.assert_allclose([doc.score for doc in results],reference_distances[expected_order],atol=2e-6)
for doc in results:
    print(json.dumps({'id':doc.id,'cosine_distance':round(doc.score,6),'text':doc.fields['text']}))

In [ ]:
context = [doc.fields['text'] for doc in results]
assert context == [documents[i] for i in expected_order]
prompt = 'Answer the question using the context below.\n\nContext:\n' + '\n'.join(context) + '\n\nQuestion:\n' + query
assert '[retrieved documents]' not in prompt and query in prompt
print(prompt)

In [ ]:
GENERATOR_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
GENERATOR_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
checkpoint = snapshot_download(GENERATOR_ID,revision=GENERATOR_REVISION,cache_dir=cache,token=False,
    allow_patterns=['config.json','generation_config.json','tokenizer.json','tokenizer_config.json',
        'special_tokens_map.json','vocab.json','merges.txt','model.safetensors'])
tokenizer = AutoTokenizer.from_pretrained(checkpoint,local_files_only=True,trust_remote_code=False)
generator = AutoModelForCausalLM.from_pretrained(checkpoint,local_files_only=True,
    trust_remote_code=False,dtype=torch.float32).eval()
inputs = tokenizer.apply_chat_template([
    {'role':'system','content':'Answer the question in one short sentence using only the supplied context. Define the concept, not merely a product name.'},
    {'role':'user','content':prompt}],add_generation_prompt=True,return_tensors='pt',return_dict=True)
assert inputs['input_ids'].shape[1] <= 512
with torch.no_grad():
    output = generator.generate(**inputs,max_new_tokens=64,do_sample=False,pad_token_id=tokenizer.eos_token_id)
continuation = output[:,inputs['input_ids'].shape[1]:]
answer = tokenizer.decode(continuation[0],skip_special_tokens=True)
assert answer.strip() and 0 < continuation.shape[1] <= 64
assert 'similar' in answer.lower() or 'semantic' in answer.lower()
print('Local model:',GENERATOR_ID,'; revision:',GENERATOR_REVISION)
print('Answer:',answer)
print('Safetensors SHA256:',hashlib.sha256((Path(checkpoint)/'model.safetensors').read_bytes()).hexdigest())

In [ ]:
# Independent exact-cosine checks for every original document as query.
for vector in embeddings:
    returned = db.query(zvec.Query(field_name='embedding',vector=vector.tolist()),topk=3,output_fields=['text'])
    distances = 1.0 - embeddings @ vector
    order = np.argsort(distances)
    assert [doc.id for doc in returned] == [f'doc_{i}' for i in order]
    np.testing.assert_allclose([doc.score for doc in returned],distances[order],atol=2e-6)
try:
    db.query(zvec.Query(field_name='embedding',vector=[0.1,0.2]),topk=2)
except (RuntimeError,ValueError,TypeError):
    pass
else:
    raise AssertionError('Wrong-dimensional query accepted')
db.close()
db = zvec.open(str(path))
reopened = db.query(zvec.Query(field_name='embedding',vector=query_vector.tolist()),topk=2,output_fields=['text'])
assert [(doc.id,doc.fields['text'])for doc in reopened] == [(doc.id,doc.fields['text'])for doc in results]
np.testing.assert_allclose([doc.score for doc in reopened],[doc.score for doc in results],atol=1e-7)
db.close()
del encoder,generator
gc.collect()
work.cleanup()
print('All original document queries match independent cosine ranking; invalid dimension rejected; durable reopen matches.')

## Original source appendix — not executable

Historical blocks are preserved verbatim; the maintained code above uses the real current API.

### Original block 1

```text
Application
     ↓
API / SDK
     ↓
Vector Database Server
```

### Original block 2

```text
Application
     ↓
Zvec Library
     ↓
Local Vector Storage
```

### Original block 3

```text
| Feature        | Traditional Vector Databases | Zvec             |
| -------------- | ---------------------------- | ---------------- |
| Deployment     | Separate service             | Embedded library |
| Infrastructure | Required                     | None             |
| Latency        | Network dependent            | Local            |
| Setup          | More complex                 | Lightweight      |
```

### Original block 4

```text
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(”all-MiniLM-L6-v2”)
documents = [
    “Zvec is an embedded vector database for AI applications.”,
    “Vector databases enable semantic similarity search.”,
    “Retrieval Augmented Generation improves LLM responses using external data.”
]
embeddings = model.encode(documents)
```

### Original block 5

```text
!pip install zvec
```

### Original block 6

```text
import zvec
db = zvec.Zvec(”knowledge.db”)
for text, vector in zip(documents, embeddings):
    db.insert(vector=vector, metadata={”text”: text})
```

### Original block 7

```text
query = “What is a vector database?”
query_vector = model.encode(query)
```

### Original block 8

```text
results = db.search(query_vector, top_k=2)
```

### Original block 9

```text
context = [r.metadata[”text”] for r in results]
```

### Original block 10

```text
Answer the question using the context below.
Context:
[retrieved documents]

Question:
What is a vector database?
```